# Kaggle Kernel: Hyperparameter Search with Optuna
This notebook runs a hyperparameter search using Optuna for a tabular competition. Results (best params, score history) are saved to output files.

In [ ]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import os, json

In [ ]:
# Load data (replace with actual paths or Kaggle Datasets API)
train = pd.read_csv('train.csv')
target = train['target']
features = train.drop(['target'], axis=1)

In [ ]:
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'num_leaves': trial.suggest_int('num_leaves', 16, 128),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100)
    }
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in kf.split(features):
        X_train, X_val = features.iloc[train_idx], features.iloc[val_idx]
        y_train, y_val = target.iloc[train_idx], target.iloc[val_idx]
        dtrain = lgb.Dataset(X_train, y_train)
        dval = lgb.Dataset(X_val, y_val, reference=dtrain)
        model = lgb.train(params, dtrain, valid_sets=[dval], verbose_eval=False, num_boost_round=100)
        preds = model.predict(X_val)
        score = mean_squared_error(y_val, preds, squared=False)
        scores.append(score)
    return np.mean(scores)

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30, timeout=60)
# Save best params and history
with open('best_params.json', 'w') as f:
    json.dump(study.best_params, f)
history = [{ 'value': t.value, 'params': t.params } for t in study.trials]
with open('search_history.json', 'w') as f:
    json.dump(history, f)